# Analisis de Produccion Minera en Peru (2018-2024)
Estudio de la distribucion regional y concentracion de la produccion de metales principales.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

In [ ]:
np.random.seed(42)

regiones = ['Ancash', 'Arequipa', 'Cajamarca', 'Cusco', 'Junin',
            'Pasco', 'Tacna', 'Moquegua', 'La Libertad', 'Apurimac', 'Ica']

metales = ['cobre', 'oro', 'plata', 'zinc', 'plomo', 'hierro', 'estano', 'molibdeno']

fechas = pd.date_range('2018-01', '2024-12', freq='MS')
n_meses = len(fechas)

In [ ]:
pesos_cobre = {
    'Ancash': 0.22, 'Arequipa': 0.20, 'Apurimac': 0.18, 'Cusco': 0.10,
    'Tacna': 0.08, 'Moquegua': 0.07, 'Junin': 0.05, 'Cajamarca': 0.04,
    'Pasco': 0.03, 'La Libertad': 0.02, 'Ica': 0.01
}

pesos_oro = {
    'Cajamarca': 0.18, 'La Libertad': 0.22, 'Arequipa': 0.12, 'Cusco': 0.10,
    'Ancash': 0.08, 'Pasco': 0.07, 'Junin': 0.06, 'Moquegua': 0.05,
    'Apurimac': 0.05, 'Tacna': 0.04, 'Ica': 0.03
}

pesos_plata = {
    'Ancash': 0.18, 'Junin': 0.16, 'Pasco': 0.14, 'La Libertad': 0.12,
    'Arequipa': 0.10, 'Cajamarca': 0.08, 'Cusco': 0.06, 'Apurimac': 0.05,
    'Tacna': 0.04, 'Moquegua': 0.04, 'Ica': 0.03
}

pesos_zinc = {
    'Ancash': 0.20, 'Junin': 0.22, 'Pasco': 0.18, 'La Libertad': 0.10,
    'Cajamarca': 0.08, 'Cusco': 0.06, 'Arequipa': 0.05, 'Apurimac': 0.04,
    'Tacna': 0.03, 'Moquegua': 0.02, 'Ica': 0.02
}

In [ ]:
def factor_covid(fecha):
    if fecha.year == 2020:
        if fecha.month in [4, 5]:
            return 0.35 + np.random.uniform(-0.05, 0.05)
        if fecha.month == 6:
            return 0.55 + np.random.uniform(-0.05, 0.05)
        if fecha.month in [3, 7]:
            return 0.75 + np.random.uniform(-0.05, 0.05)
        if fecha.month in [8, 9]:
            return 0.88 + np.random.uniform(-0.03, 0.03)
    return 1.0

def generar_produccion(fechas, base_total, pesos, ruido_pct=0.08):
    registros = []
    for fecha in fechas:
        covid = factor_covid(fecha)
        tendencia = 1 + 0.015 * (fecha.year - 2018)
        estacional = 1 + 0.03 * np.sin(2 * np.pi * fecha.month / 12)
        for region, peso in pesos.items():
            valor = base_total * peso * tendencia * estacional * covid
            valor *= (1 + np.random.normal(0, ruido_pct))
            registros.append({'fecha': fecha, 'region': region, 'produccion': max(valor, 0)})
    return registros

registros = []
configs = [
    ('cobre', 48000, pesos_cobre),
    ('oro', 5_500_000, pesos_oro),
    ('plata', 320, pesos_plata),
    ('zinc', 100000, pesos_zinc),
]

pesos_generico = {r: 1/len(regiones) for r in regiones}
configs += [
    ('plomo', 25000, pesos_generico),
    ('hierro', 900000, {'Ica': 0.65, 'Arequipa': 0.15, 'Ancash': 0.08, 'Junin': 0.04,
                        'Pasco': 0.02, 'Cusco': 0.02, 'La Libertad': 0.01, 'Cajamarca': 0.01,
                        'Tacna': 0.01, 'Moquegua': 0.005, 'Apurimac': 0.005}),
    ('estano', 2500, {'Puno': 0, 'Ancash': 0.3, 'Pasco': 0.25, 'Junin': 0.2,
                      'La Libertad': 0.1, 'Arequipa': 0.05, 'Cajamarca': 0.04,
                      'Cusco': 0.03, 'Tacna': 0.01, 'Moquegua': 0.01, 'Apurimac': 0.01}),
    ('molibdeno', 2800, pesos_cobre),
]

for metal, base, pesos in configs:
    datos = generar_produccion(fechas, base, pesos)
    for d in datos:
        d['metal'] = metal
    registros.extend(datos)

df = pd.DataFrame(registros)
df['anio'] = df['fecha'].dt.year
df['mes'] = df['fecha'].dt.month
print(f'Registros: {len(df):,} | Metales: {df.metal.nunique()} | Regiones: {df.region.nunique()}')

## 1. Produccion mensual de cobre por region (Top 5)

In [ ]:
df_cobre = df[df['metal'] == 'cobre'].copy()
top5_cobre = df_cobre.groupby('region')['produccion'].sum().nlargest(5).index.tolist()

pivot_cobre = df_cobre[df_cobre['region'].isin(top5_cobre)].pivot_table(
    index='fecha', columns='region', values='produccion', aggfunc='sum'
)[top5_cobre]

colores_region = ['#1b4332', '#2d6a4f', '#40916c', '#52b788', '#95d5b2']

fig, ax = plt.subplots(figsize=(12, 5))
ax.stackplot(pivot_cobre.index, *[pivot_cobre[c] for c in pivot_cobre.columns],
             labels=pivot_cobre.columns, colors=colores_region, alpha=0.85)

ax.axvspan(pd.Timestamp('2020-03-15'), pd.Timestamp('2020-07-15'),
           alpha=0.12, color='red', label='Impacto COVID')

ax.set_title('Produccion Mensual de Cobre - Top 5 Regiones (TMF)', fontweight='bold', pad=12)
ax.set_ylabel('TMF / mes')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
ax.legend(loc='upper left', framealpha=0.9, fontsize=8)
ax.set_xlim(pivot_cobre.index[0], pivot_cobre.index[-1])
sns.despine(left=True)
plt.tight_layout()
plt.show()

## 2. Correlacion entre produccion de metales

In [ ]:
prod_mensual = df.pivot_table(index='fecha', columns='metal', values='produccion', aggfunc='sum')
corr = prod_mensual.corr()

mascara = np.triu(np.ones_like(corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, mask=mascara, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.8,
            cbar_kws={'shrink': 0.75, 'label': 'Correlacion'},
            ax=ax)
ax.set_title('Correlacion entre Produccion de Metales', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

## 3. Indice HHI de concentracion por metal

In [ ]:
def calcular_hhi(grupo):
    participaciones = grupo / grupo.sum() * 100
    return (participaciones ** 2).sum()

prod_region_metal = df.groupby(['metal', 'region'])['produccion'].sum().reset_index()
hhi = prod_region_metal.groupby('metal')['produccion'].apply(calcular_hhi).sort_values(ascending=False)

colores_hhi = ['#c1121f' if v > 2500 else '#e07a5f' if v > 1500 else '#457b9d' for v in hhi.values]

fig, ax = plt.subplots(figsize=(10, 5))
barras = ax.barh(hhi.index, hhi.values, color=colores_hhi, edgecolor='white', height=0.6)

ax.axvline(2500, color='#c1121f', ls='--', lw=1, alpha=0.7)
ax.text(2550, len(hhi)-0.5, 'Alta concentracion', color='#c1121f', fontsize=8, va='center')
ax.axvline(1500, color='#e07a5f', ls='--', lw=1, alpha=0.7)
ax.text(1550, len(hhi)-1.5, 'Moderada', color='#e07a5f', fontsize=8, va='center')

for barra, valor in zip(barras, hhi.values):
    ax.text(barra.get_width() + 30, barra.get_y() + barra.get_height()/2,
            f'{valor:.0f}', va='center', fontsize=9)

ax.set_title('Indice Herfindahl-Hirschman por Metal', fontweight='bold', pad=12)
ax.set_xlabel('HHI')
sns.despine()
plt.tight_layout()
plt.show()

## 4. Treemap interactivo: Participacion por metal y region

In [ ]:
df_tree = prod_region_metal.copy()
total_metal = df_tree.groupby('metal')['produccion'].transform('sum')
df_tree['pct'] = (df_tree['produccion'] / total_metal * 100).round(1)
df_tree = df_tree[df_tree['pct'] >= 1.0]

fig_tree = px.treemap(
    df_tree, path=['metal', 'region'], values='produccion',
    color='pct', color_continuous_scale='Viridis',
    title='Participacion en Produccion Minera por Metal y Region',
    hover_data={'pct': ':.1f'}
)
fig_tree.update_layout(
    margin=dict(t=50, l=10, r=10, b=10),
    coloraxis_colorbar_title='% del metal',
    font_size=11
)
fig_tree.show()

## 5. Variacion interanual del cobre con anotacion COVID

In [ ]:
cobre_mensual = df_cobre.groupby('fecha')['produccion'].sum().reset_index()
cobre_mensual = cobre_mensual.sort_values('fecha')
cobre_mensual['yoy'] = cobre_mensual['produccion'].pct_change(12) * 100
cobre_mensual = cobre_mensual.dropna(subset=['yoy'])

fig_yoy = go.Figure()
fig_yoy.add_trace(go.Scatter(
    x=cobre_mensual['fecha'], y=cobre_mensual['yoy'],
    mode='lines', name='Var. YoY %',
    line=dict(color='#2d6a4f', width=2),
    fill='tozeroy', fillcolor='rgba(45,106,79,0.15)',
    hovertemplate='%{x|%b %Y}: %{y:.1f}%<extra></extra>'
))

fig_yoy.add_hline(y=0, line_dash='dot', line_color='gray', line_width=1)

fig_yoy.add_vrect(
    x0='2020-03-15', x1='2020-07-15',
    fillcolor='red', opacity=0.08, line_width=0,
    annotation_text='Cuarentena nacional',
    annotation_position='top left',
    annotation_font_size=10
)

fig_yoy.update_layout(
    title='Variacion Interanual de Produccion de Cobre (%)',
    yaxis_title='Variacion YoY (%)',
    template='plotly_white',
    height=420,
    margin=dict(t=60, b=40),
    font_size=11
)
fig_yoy.show()

## Observaciones

- La produccion de cobre esta concentrada en Ancash, Arequipa y Apurimac (HHI alto).
- El oro presenta distribucion mas equitativa entre regiones productoras.
- El impacto COVID genero caidas de 40-65% en abril-mayo 2020, con recuperacion en el segundo semestre.
- La correlacion entre metales base (cobre, zinc, plomo) es consistente por compartir regiones productoras.